# Desagrupamiento o Declustering

Aplicación de método de las celdas

In [1]:
# Importamos las librerías necesarias
import os
import sys
import random
import numpy as np
import pandas as pd

In [2]:
# Llamamos nuestra base de datos de ejemplo
name = "Copper_Gold_Drillholes.csv"
df = pd.read_csv(name, sep=",")
print(f"Base de datos '{name}' cargada con éxito. Número de filas: {len(df)}")  

Base de datos 'Copper_Gold_Drillholes.csv' cargada con éxito. Número de filas: 2376


In [3]:
# Inputs del usuario
def header():
    print("\n" + "=" * 62)
    print("   DECLUSTERING  -  Metodo de Celdas (Cell Declustering)")
    print("=" * 62 + "\n")


def ask(prompt, default=None):
    suffix = f" [{default}]" if default is not None else ""
    raw = input(f"  {prompt}{suffix}: ").strip()
    return raw if raw else (str(default) if default is not None else "")


def ask_column(prompt, columns, default=None):
    print(f"\n  Columnas disponibles: {', '.join(columns)}")
    while True:
        val = ask(prompt, default)
        if val in columns:
            return val
        print(f"  [!] '{val}' no existe. Opciones: {', '.join(columns)}")


def ask_multi_column(prompt, columns, default=None):
    print(f"\n  Columnas numericas disponibles: {', '.join(columns)}")
    while True:
        raw  = ask(prompt, default)
        vals = [v.strip() for v in raw.split(",") if v.strip()]
        bad  = [v for v in vals if v not in columns]
        if vals and not bad:
            return vals
        print(f"  [!] No reconocidas: {bad}. Opciones: {', '.join(columns)}")


def ask_float(prompt, default=None):
    while True:
        try:
            return float(ask(prompt, default))
        except ValueError:
            print("  [!] Ingresa un numero valido.")


def ask_int(prompt, default=None):
    while True:
        try:
            return int(ask(prompt, default))
        except ValueError:
            print("  [!] Ingresa un entero valido.")

In [ ]:
# Aplicación de algoritmo de declusterización por celdas
def decluster_cells(x_coords, y_coords, z_coords,
                    xcellsize, ycellsize, zcellsize,
                    niterations):
    """
    Calcula los pesos de declustering usando el Metodo de Celdas.

    Para cada iteracion se desplaza aleatoriamente el origen de la
    grilla dentro de una celda, se asigna cada muestra a una celda
    y el peso de cada muestra es la inversa del numero de muestras
    en su misma celda. El peso final es el promedio de todas las
    iteraciones.

    Args:
        x_coords, y_coords, z_coords : arrays de coordenadas
        xcellsize, ycellsize, zcellsize : tamano de celda por eje
        niterations : numero de iteraciones aleatorias

    Returns:
        np.ndarray : array de pesos (mismo largo que coords)
    """
    xo = np.min(x_coords) - 0.01
    yo = np.min(y_coords) - 0.01
    zo = np.min(z_coords) - 0.01

    all_weights = []

    for _ in range(niterations):
        # Origen con componente aleatoria por iteracion
        x_origin = xo - random.uniform(0, 1) * xcellsize
        y_origin = yo - random.uniform(0, 1) * ycellsize
        z_origin = zo - random.uniform(0, 1) * zcellsize

        # Numero de celdas en X e Y
        xn_cell = int((np.max(x_coords) - x_origin) / xcellsize) + 1
        yn_cell = int((np.max(y_coords) - y_origin) / ycellsize) + 1

        # Indice de celda por muestra
        xi_cell = (((x_coords - x_origin) / xcellsize) + 1).astype(int)
        yi_cell = (((y_coords - y_origin) / ycellsize) + 1).astype(int)
        zi_cell = (((z_coords - z_origin) / zcellsize) + 1).astype(int)

        # Indice unico de celda 3D (aplanado)
        i_cell = xi_cell + (yi_cell - 1) * xn_cell + (zi_cell - 1) * xn_cell * yn_cell - 1

        # Conteo de muestras por celda
        cell_counts = np.bincount(i_cell)

        # Peso = 1 / N_muestras_en_celda
        weights = [1.0 / cell_counts[int(i_cell[k])]
                   for k in range(len(x_coords))]
        all_weights.append(weights)

    return np.average(all_weights, axis=0)


def decluster_cells_properties(df, xaxis, yaxis, zaxis, properties,
                                xcellsize, ycellsize, zcellsize,
                                niterations, filter_val):
    """
    Calcula pesos de declustering para muestreo heterotopico
    (una o mas propiedades con datos faltantes independientes).

    Aplica un filtro previo (valores <= filter_val se tratan como NaN)
    y promedia los pesos de cada propiedad para obtener un unico
    vector de pesos por muestra.

    Args:
        df         : DataFrame con coordenadas y propiedades
        filter_val : umbral de filtro (valores <= se ignoran)

    Returns:
        np.ndarray : pesos finales (NaN donde todas las props son NaN)
    """
    n_datum = len(df)

    # DataFrame filtrado: valores <= filter_val -> NaN
    df_filt = df[[xaxis, yaxis, zaxis] + properties].copy()
    for prop in properties:
        df_filt[prop] = pd.to_numeric(df_filt[prop], errors="coerce")
        df_filt.loc[df_filt[prop] <= filter_val, prop] = np.nan

    all_prop_weights = []

    for prop in properties:
        prop_arr = np.array(df_filt[prop])
        mask     = ~np.isnan(prop_arr)

        if mask.sum() == 0:
            print(f"  [!] '{prop}': todos los valores son NaN tras el filtro. Se omite.")
            continue

        x_prop = np.array(df_filt[xaxis][mask])
        y_prop = np.array(df_filt[yaxis][mask])
        z_prop = np.array(df_filt[zaxis][mask])

        weights_prop = decluster_cells(
            x_prop, y_prop, z_prop,
            xcellsize, ycellsize, zcellsize,
            niterations).tolist()

        # Expandir pesos al largo total del dataframe (NaN donde no hay dato)
        indexes      = [i for i, x in enumerate(mask) if x]
        weights_full = [np.nan] * n_datum
        for i, idx in enumerate(indexes):
            weights_full[idx] = weights_prop[i]

        all_prop_weights.append(weights_full)

    return np.nanmean(all_prop_weights, axis=0), df_filt


def compute_stats(df_filt, df_weights, properties, weight_col):
    """
    Calcula estadisticas declusterizadas por propiedad:
    N, Media, Varianza, Desviacion Estandar.

    Returns:
        pd.DataFrame con una fila por propiedad
    """
    rows = []
    for prop in properties:
        values  = np.array(df_filt[prop])
        mask    = ~np.isnan(values)
        n_data  = int(mask.sum())
        weights = np.array(df_weights[weight_col]) * (mask * 1.0)

        w_sum       = np.nansum(weights)
        declus_mean = np.nansum(weights * values) / w_sum
        declus_var  = np.nansum(weights * (values - declus_mean) ** 2) / w_sum
        declus_std  = np.sqrt(declus_var)

        rows.append({
            "Property"       : prop,
            "N"              : n_data,
            "Declus Mean"    : round(declus_mean, 4),
            "Declus Variance": round(declus_var,  4),
            "Declus Std Dev" : round(declus_std,  4),
        })

    return pd.DataFrame(rows)

In [ ]:
# Flujo de trabajo 
def main(df):
    header()

    columns  = list(df.columns)
    num_cols = [c for c in columns if pd.api.types.is_numeric_dtype(df[c])]

    print(f"  DataFrame cargado: {len(df):,} filas | {len(columns)} columnas")
    print(f"  Columnas: {', '.join(columns)}\n")

    # 1. Coordenadas
    print("[ 1 / 4 ]  Coordenadas espaciales")
    xaxis = ask_column("Columna X (Este/Easting)",          num_cols)
    yaxis = ask_column("Columna Y (Norte/Northing)",        num_cols)
    zaxis = ask_column("Columna Z (Profundidad/Elevacion)", num_cols)
    df[[xaxis, yaxis, zaxis]] = df[[xaxis, yaxis, zaxis]].apply(
        pd.to_numeric, errors="coerce")

    # 2. Variables a declusterar
    print("\n[ 2 / 4 ]  Variables a declusterar")
    prop_candidates = [c for c in num_cols if c not in [xaxis, yaxis, zaxis]]
    properties = ask_multi_column(
        "Variable(s)  [separar con coma]", prop_candidates)
    df[properties] = df[properties].apply(pd.to_numeric, errors="coerce")

    # 3. Tamano de celdas y filtro
    print("\n[ 3 / 4 ]  Parametros de celda y filtro")
    print("  (El tamano de celda define la zona de influencia de cada muestra.")
    print("   Usar aproximadamente el espaciado tipico entre sondajes.)")
    xcellsize  = ask_float("Tamano de celda en X (metros)", default=50)
    ycellsize  = ask_float("Tamano de celda en Y (metros)", default=50)
    zcellsize  = ask_float("Tamano de celda en Z (metros)", default=10)
    filter_val = ask_float("Filtro — excluir valores <=  (usar -9999 para no filtrar)", default=-9999)

    # 4. Iteraciones  (ultimo input -> arranca solo)
    print("\n[ 4 / 4 ]  Iteraciones del metodo de celdas")
    print("  (Mayor numero de iteraciones = pesos mas estables.")
    print("   Recomendado: entre 50 y 500.)")
    niterations = ask_int("Numero de iteraciones", default=100)

    # Procesamiento inmediato
    print("\n" + "-" * 62)
    print(f"  Calculando pesos de declustering...")
    print(f"  Variables   : {properties}")
    print(f"  Celda XYZ   : {xcellsize} x {ycellsize} x {zcellsize} m")
    print(f"  Filtro      : valores <= {filter_val} excluidos")
    print(f"  Iteraciones : {niterations}")
    print("-" * 62 + "\n")

    weights, df_filt = decluster_cells_properties(
        df, xaxis, yaxis, zaxis, properties,
        xcellsize, ycellsize, zcellsize,
        niterations, filter_val)

    df["Weights"] = weights

    # Estadisticas
    df_stats = compute_stats(df_filt, df, properties, "Weights")

    # Guardar resultados
    base_name   = "declustered"
    out_data    = f"{base_name}_data.csv"
    out_stats   = f"{base_name}_stats.csv"

    df.to_csv(out_data,  index=False)
    df_stats.to_csv(out_stats, index=False)

    # Mostrar resultados en consola
    print("  ESTADISTICAS DECLUSTERIZADAS:")
    print("  " + "-" * 55)
    print(df_stats.to_string(index=False))
    print()
    print(f"  [OK] Datos + pesos guardados  -> {out_data}")
    print(f"  [OK] Estadisticas guardadas   -> {out_stats}")
    print("-" * 62 + "\n")


# -------------------------------------------------------------
# Comienzo del programa
# -------------------------------------------------------------
if __name__ == "__main__":
    main(df)



   DECLUSTERING  -  Metodo de Celdas (Cell Declustering)

  DataFrame cargado: 2,376 filas | 6 columnas
  Columnas: X, Y, Z, Cu, Au, Rock_Type

[ 1 / 4 ]  Coordenadas espaciales

  Columnas disponibles: X, Y, Z, Cu, Au, Rock_Type

  Columnas disponibles: X, Y, Z, Cu, Au, Rock_Type

  Columnas disponibles: X, Y, Z, Cu, Au, Rock_Type

[ 2 / 4 ]  Variables a declusterar

  Columnas numericas disponibles: Cu, Au, Rock_Type

[ 3 / 4 ]  Parametros de celda y filtro
  (El tamano de celda define la zona de influencia de cada muestra.
   Usar aproximadamente el espaciado tipico entre sondajes.)

[ 4 / 4 ]  Iteraciones del metodo de celdas
  (Mayor numero de iteraciones = pesos mas estables.
   Recomendado: entre 50 y 500.)

--------------------------------------------------------------
  Calculando pesos de declustering...
  Variables   : ['Cu', 'Au']
  Celda XYZ   : 25.0 x 25.0 x 5.0 m
  Filtro      : valores <= -9999.0 excluidos
  Iteraciones : 100
-------------------------------------------

In [ ]:
declus = "declustered_data.csv"
declus_stats = "declustered_stats.csv"
df_declus = pd.read_csv(declus, sep=",")
df_stats = pd.read_csv(declus_stats, sep=",")

Base de datos 'Copper_Gold_Drillholes.csv' cargada con éxito. Número de filas: 2376
Base de datos 'Copper_Gold_Drillholes.csv' cargada con éxito. Número de filas: 2


In [7]:
df_declus

,X,Y,Z,Cu,Au,Rock_Type,Weights
0,193.0,528.6,39.0,0.12,0.000,4,1.0000
1,335.1,38.0,97.0,0.13,0.028,4,0.8550
2,250.7,593.4,36.0,0.13,0.027,4,1.0000
3,275.8,517.2,86.1,0.19,0.000,4,0.8225
4,256.1,529.1,61.8,0.19,0.039,4,1.0000
...,...,...,...,...,...,...,...
2371,391.6,295.2,19.5,0.48,4.579,54,1.0000
2372,382.9,295.2,11.2,0.59,1.757,54,0.9925
2373,359.7,294.9,24.7,0.88,2.722,54,1.0000
2374,358.7,295.0,12.7,1.14,19.045,54,0.9925


In [8]:
df_stats

,Property,N,Declus Mean,Declus Variance,Declus Std Dev
0,Cu,2376,1.0485,0.4096,0.6400
1,Au,2376,3.9137,51.0380,7.1441
